# 9.5 Reading unformatted data: the read command

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

The read command provides a particularly simple way of getting values into AMPL,
given that the values you need are listed in a regular order in a file. The file must be
unformatted in the sense that it contains nothing except the values to be read — no set or
parameter names, no colons or := operators.

In its simplest form, read specifies a list of parameters and a file from which their
values are to be read. The values in the file are assigned to the entries in the list in the
order that they appear. For example, if you want to read the number of weeks and the
hours available each week for our simple production `model` ([Figure 4-4](../04/4_3_a_model_of_production_and_transportation.ipynb#fig-4-4)),

```ampl
param T > 0;
param avail {1..T} >= 0;
```

from a file week_data.txt containing

```ampl
4
40 40 32 40
```

then you can give the command

```ampl
read T, avail[1], avail[2], avail[3], avail[4] <week_data.txt;
```

Or you can use an indexing expression to say the same thing more concisely and generally
:

```ampl
read T, {t in 1..T} avail[t] <week_data.txt;
```

The notation < filename specifies the name of a file for reading. (Analogously, > indicates
writing to a file; see A.15.)
In general, the read command has the form

```ampl
read item-list < filename ;
```

with the item-list being a comma-separated list of items that may each be any of the following
:

```ampl
parameter
{ indexing } parameter
{ indexing } ( item-list )
```

The first two are used in our example above, while the third allows for the same indexing
to be applied to several items. Using the same production example, to read in values for

```ampl
param prodcost {PROD} >= 0;
param invcost {PROD} >= 0;
param revenue {PROD,1..T} >= 0;
```

from a file organized by parameters, you could read each parameter separately:

```ampl
read {p in PROD} prodcost[p] < cost_data;
read {p in PROD} invcost[p] < cost_data;
read {p in PROD, t in 1..T} revenue[p,t] < cost_data;
```

reading from file cost_data first all the production costs, then all the inventory costs,
and then all the revenues.

If the `data` were organized by product instead, you could say

```ampl
read {p in PROD}
   (prodcost[p], invcost[p], {t in 1..T} revenue[p,t])
      <cost_data;
```

to read the production and inventory costs and the revenues for the first product, then for
the second product, and so forth.

A parenthesized item-list may itself contain parenthesized item-lists, so that if you
also want to read

```ampl
param market {PROD,1..T} >= 0;
```

from the same file at the same time, you could say

```ampl
read {p in PROD} (prodcost[p], invcost[p],
   {t in 1..T} (revenue[p,t], market[p,t])) <cost_data;
```

in which case for each product you would read the two costs as before, and then for each
week the product's revenue and market demand.

As our descriptions suggest, the form of a read statement's item-list depends on how
the `data` values are ordered in the file. When you are reading `data` indexed over sets of
strings that, like PROD, are not inherently ordered, then the order in which values are read
is the order in which AMPL is internally representing them. If the members of the set
came directly from a set `data` statement, then the ordering will be the same as in the `data`
statement. Otherwise, it is best to put an ordered or ordered by phrase in the
model's set declaration to ensure that the ordering is always what you expect;
see [Section 5.6](../05/5_6_ordered_sets.ipynb) for more about ordered sets.

An alternative that avoids knowing the order of the members in a set is to specify
them explicitly in the file that is read. As an example, consider how you might use a
read statement rather than a `data` statement to get the values from the cost parameter
of Section 9.4 that was defined as

```ampl
param cost {ORIG,DEST,PROD} >= 0, default 9999;
```

You could set up the read statement as follows:

```ampl
param   ntriples integer;
param   ic symbolic in ORIG;
param   jc symbolic in DEST;
param   kc symbolic in PROD;
read ntriples, {1..ntriples}
   (ic, jc, kc, cost[ic,jc,kc]) <cost_data;
```

The corresponding file cost_data must begin something like this:

```ampl
18
CLEV FRA bands 27
PITT FRA bands 24
CLEV FRA coils 23
...
```

with 15 more entries needed to give all 18 `data` values shown in the Section 9.4 example.

Strings in a file for the read command that include any character other than letters,
digits, underscores, period, + and - must be quoted, just as for `data` mode. However, the
read statement itself is interpreted in `model` mode, so if the statement refers to any particular
string, as in, say,

```ampl
read {t in 1..T} revenue ["bands",t];
```

that string must be quoted. The filename following < need not be quoted unless it contains
spaces, semicolons, or nonprinting characters.

If a read statement contains no < filename, values are read from the current input
stream. Thus if you have typed the read command at an AMPL prompt, you can type
the values at subsequent prompts until all of the listed items have been assigned values.
For example:

```ampl
ampl:    read T, {t in 1..T} avail[t];
ampl?    4
ampl?    40 40 32 40
ampl:    display avail;
avail    [*] :=
1 40       2 40  3 32   4 40
;
```

The prompt changes from ampl? back to ampl: when all the needed input has been
read.

The filename "-" (a literal minus sign) is taken as the standard input of the AMPL
process; this is useful for providing input interactively.

Further uses of read within AMPL scripts, to read values directly from script files or
to prompt users for values at the command line, are described in Chapter 13.

All of our examples assume that underlying sets such as ORIG and PROD have
already been assigned values, through `data` statements as described earlier in this chapter,
or through other means such as database access or assignment to be described in later
chapters. Thus the read statement would normally supplement rather than replace other
input commands. It is particularly useful in handling long files of `data` that are generated
for certain parameters by programs outside of AMPL.